In [1]:
import os
import re
import glob
import numpy as np
import pandas as pd
import astropy.units as u
import matplotlib.pyplot as plt

from pathlib import Path
from astropy.io import fits
from astropy.wcs import WCS
from astroquery.simbad import Simbad
from matplotlib.patches import Rectangle
from astroquery.hips2fits import hips2fits
from astropy.coordinates import SkyCoord, Angle

font = {'family' : 'gill sans',
         'size'   : 12,
         'serif':  'cmr10'
         }

plt.rc('font', **font)
plt.rc('axes', unicode_minus = False)

In [2]:
hdul = fits.open('/Users/billyli/Documents/float-for-morrow/LMC/30 Doradus/RMC 140/E140M_1425_0.2x0.2_2670.000_16272/oec159010_x1d.fits')
hdul.info()
R = float(hdul[0].header["SPECRES"])
print(R)

Filename: /Users/billyli/Documents/float-for-morrow/LMC/30 Doradus/RMC 140/E140M_1425_0.2x0.2_2670.000_16272/oec159010_x1d.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     298   ()      
  1  SCI           1 BinTableHDU    158   44R x 19C   [1I, 1I, 1024D, 1024E, 1024E, 1024E, 1024E, 1024E, 1024E, 1024I, 1E, 1E, 1I, 1E, 1E, 1E, 1E, 1024E, 1E]   
45800.0


In [3]:
main_directory = Path().resolve()

In [4]:
def snr_from_spectra(path, Cenwave, Instrument):

    if Cenwave == 1096:

        wave = 1100

    elif Cenwave == 1291 or Cenwave == 1327:

        wave = 1292

    elif Cenwave == 1611 or Cenwave == 1577 or Cenwave == 1623:

        wave = 1630

    elif Cenwave == 1425:

        wave = 1420

    elif Cenwave == 1978:

        wave = 2002

    elif Cenwave == 2707:

        wave = 2800
    
    with fits.open(path) as hdul:
        data = hdul[1].data

        wl = data["WAVELENGTH"]
        flux = data["FLUX"]
        err = data["ERROR"]

        header = hdul[0].header
        R = float(header["SPECRES"])
        dl_pix = float(np.nanmedian(np.diff(wl)))
        dl_res = np.nanmedian(wl) / R

        npix = dl_res / dl_pix
        
        mask = (np.abs(wl - wave) < 0.5) & np.isfinite(flux) & np.isfinite(err) & (err > 0)
        snr = flux[mask] / err[mask] * np.sqrt(npix)

        return float(np.median(snr))

def query_simbad(target_name):
    result = Simbad().query_object(target_name)
    ra_deg = float(result["ra"][0])
    dec_deg = float(result["dec"][0])
    coord = SkyCoord(ra = ra_deg * u.deg, dec = dec_deg * u.deg, frame = "icrs")
    ra_str = coord.ra.to_string(unit = u.hourangle, sep = " ", pad = True, precision = 3, alwayssign = False)
    dec_str = coord.dec.to_string(unit = u.deg, sep = " ", pad = True, precision = 2, alwayssign = True)
    if dec_str.startswith("+"):
        dec_str = dec_str[1:]
    return f"{ra_str}", f"{dec_str}"

In [5]:
def parse_directory(target_directory, dirname, galaxy_name, region_name, target_name, angular_separation, target_id): # different with some other functions I wrote
    
    parts = dirname.split("_")

    Grating = parts[0]
    Cenwave = int(parts[1])
    T_exp = str(parts[3])
    PID = int(parts[4])

    RA, DEC = query_simbad(target_name)
    
    def get_abundance(element):
        file_path = os.path.join(target_directory, f"{element}.txt")
        if os.path.exists(file_path):
            with open(file_path, 'r') as f:
                lines = f.readlines()
                if len(lines) >= 2:
                    try:
                        val = float(lines[0].strip().split()[-1]) 
                        err = float(lines[1].strip().split()[-1])
                        return val, err
                    except (ValueError, IndexError):
                        pass
        return None, None

    h_abundance, h_err = get_abundance("NHI")
    s_abundance, s_err = get_abundance("logNS")
    fe_abundance, fe_err = get_abundance("logNFe")
    twelve_plus_s, twelve_plus_s_err = get_abundance("S")
    twelve_plus_fe, twelve_plus_fe_err = get_abundance("Fe")

    if Grating.startswith("G"):
        
        Instrument = "COS"
        LP = parts[2]

        for file in os.listdir(os.path.join(target_directory, dirname)):
            if file.endswith("_x1dsum.fits"):
                path = os.path.join(target_directory, dirname, file)

        snr = snr_from_spectra(path, Cenwave, Instrument)

        return {
            # "Galaxy": galaxy_name, 
            "ID": target_id,
            "Target": target_name, 
            "NHI": f"{h_abundance:.3f} \u00b1 {h_err:.3f}",
            "NS": f"{s_abundance:.3f} \u00b1 {s_err:.3f}",
            "NFe": f"{fe_abundance:.3f} \u00b1 {fe_err:.3f}",
            "S": f"{twelve_plus_s:.3f} \u00b1 {twelve_plus_s_err:.3f}",
            "Fe": f"{twelve_plus_fe:.3f} \u00b1 {twelve_plus_fe_err:.3f}"
        }
        
    else:
        
        Instrument = "STIS"
        Aperture = parts[2]

        path = sorted(glob.glob(os.path.join(target_directory, dirname, "*_x1d.fits")))[0]
        snr = snr_from_spectra(path, Cenwave, Instrument)
        
        return {
            # "Galaxy": galaxy_name, 
            "ID": target_id,
            "Target": target_name, 
            "NHI": f"{h_abundance:.3f} \u00b1 {h_err:.3f}",
            "NS": f"{s_abundance:.3f} \u00b1 {s_err:.3f}",
            "NFe": f"{fe_abundance:.3f} \u00b1 {fe_err:.3f}",
            "S": f"{twelve_plus_s:.3f} \u00b1 {twelve_plus_s_err:.3f}",
            "Fe": f"{twelve_plus_fe:.3f} \u00b1 {twelve_plus_fe_err:.3f}"
        }

In [6]:
records = []

for galaxy_name in os.listdir(main_directory):
    galaxy_path = main_directory / galaxy_name
    if galaxy_path.name.startswith(".") or not galaxy_path.is_dir():
        continue
        
    for region_name in os.listdir(galaxy_path):
        region_path = galaxy_path / region_name
        if region_path.name.startswith(".") or not region_path.is_dir():
            continue

        region = Simbad.query_object(region_name)
        ra_region = region["ra"][0]
        dec_region = region["dec"][0]

        if region_name == "SMC N88A": 

            coord_region = SkyCoord('01h24m08.3s', '−73d09m04.6s', frame = 'icrs')

        if region_name == "SMC N90": 

            coord_region = SkyCoord('01h29m36.1s', '−73d33m51.9s', frame = 'icrs')

        if region_name == "SMC N66A": 

            coord_region = SkyCoord('00h59m14.3s', '-72d11m02.8s', frame = 'icrs')

        if region_name == "SMC N81": 

            coord_region = SkyCoord('01h09m12.8s', '−73d11m36.90s', frame = 'icrs')

        if region_name == "LMC N11B": 

            coord_region = SkyCoord('04h56m46.9s', '−66d24m37.9s', frame = 'icrs')

        if region_name == "30 Doradus": 

            coord_region = SkyCoord('05h38m53.6s', '-69d05m07.4s', frame = 'icrs')

        index = 1
        
        for target_name in sorted(os.listdir(region_path)):

            if region_name == "SMC N88A": 

                code_prefix = "C"
    
            if region_name == "SMC N90": 
    
                code_prefix = "D"
    
            if region_name == "SMC N66A": 
    
                code_prefix = "A"
    
            if region_name == "SMC N81": 
    
                code_prefix = "B"
    
            if region_name == "LMC N11B": 
    
                code_prefix = "F"
    
            if region_name == "30 Doradus": 
    
                code_prefix = "E"
            
            target_path = region_path / target_name
            if target_path.name.startswith("unused") or target_path.name.startswith(".") or not target_path.is_dir():
                continue

            target_id = code_prefix + str(index)

            index += 1

            code_file = target_path / "code.txt"
            with open(code_file, "w") as f:
                f.write(target_id)

            target = Simbad.query_object(target_name)
            ra_target = target["ra"][0]
            dec_target = target["dec"][0]
            coord_target = SkyCoord(ra_target, dec_target, unit = (u.deg, u.deg), frame = "icrs")

            angular_separation = coord_region.separation(coord_target).arcmin

            spectra_directories = sorted([d for d in os.listdir(target_path) if d.startswith("G") or d.startswith("E")])

            i = 0

            for d in spectra_directories:
                if i == 0:
                    info = parse_directory(target_path, d, galaxy_name, region_name, target_name, angular_separation, target_id)
                    records.append(info)
                    i = 1

df = pd.DataFrame.from_records(records)

In [7]:
df

,ID,Target,NHI,NS,NFe,S,Fe
0,C1,2dFS 3694,20.779 ± 0.020,15.638 ± 0.117,14.578 ± 0.025,6.857 ± 0.119,5.797 ± 0.032
1,D1,SK 183,20.595 ± 0.074,15.353 ± 0.013,14.740 ± 0.022,6.756 ± 0.075,6.143 ± 0.077
2,A1,Cl* NGC 346 ELS 22,21.176 ± 0.017,15.567 ± 0.013,15.114 ± 0.138,6.390 ± 0.022,5.937 ± 0.139
3,A2,Cl* NGC 346 ELS 50,20.816 ± 0.040,15.403 ± 0.011,14.694 ± 0.010,6.585 ± 0.042,5.876 ± 0.042
4,A3,Cl* NGC 346 ELS 51,20.905 ± 0.033,15.482 ± 0.010,14.866 ± 0.261,6.576 ± 0.035,5.960 ± 0.263
5,A4,Cl* NGC 346 ELS 7,21.096 ± 0.022,15.602 ± 0.018,15.218 ± 0.081,6.505 ± 0.029,6.121 ± 0.084
6,A5,Cl* NGC 346 MPG 782,20.043 ± 0.157,15.399 ± 0.007,14.736 ± 0.052,7.348 ± 0.158,6.685 ± 0.166
7,A6,Cl* NGC 346 NMC 17,21.288 ± 0.016,15.791 ± 0.035,15.170 ± 0.116,6.502 ± 0.039,5.881 ± 0.117
8,A7,Cl* NGC 346 NMC 28,21.030 ± 0.026,15.567 ± 0.021,14.749 ± 0.009,6.536 ± 0.034,5.718 ± 0.028
9,A8,Cl* NGC 346 SSN 25,21.487 ± 0.019,15.686 ± 0.027,14.976 ± 0.099,6.199 ± 0.033,5.489 ± 0.101


In [8]:
def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split('([0-9]+)', str(s))]

df = df.sort_values(by = "ID", key = lambda x: x.map(natural_sort_key))

In [9]:
df.to_csv("table2.csv", index = False)

In [10]:
def df_to_latex(df):
    lines = []
    prev_target = None

    for _, row in df.iterrows():
        curr_target = row["Target"]
        is_lmc = str(row.iloc[0]).startswith(('E', 'F'))
        
        formatted_values = []
        for i, v in enumerate(row):
            if pd.isna(v) or v == "":
                formatted_values.append("")
            elif isinstance(v, float):
                formatted_values.append(f"{v:.2f}")
            else:
                s = str(v)
                if is_lmc and i == 3 or is_lmc and i == 5:
                    val_only = s.split("\u00b1")[0].strip()
                    s = f"$>{val_only}$"
                elif "\u00b1" in s:
                    s = f"${s.replace('\u00b1', r'\pm')}$"
                if "0.000" in s:
                    s = f"{s.replace('0.000', '0.001')}"
                formatted_values.append(s)

        line = " & ".join(formatted_values) + r" \\"
        lines.append(line)
        prev_target = curr_target if curr_target != "" else prev_target

    return "\n".join(lines)

In [11]:
body = df_to_latex(df)

latex = rf"""
\startdata
{body}
\enddata
\label{{tab:abundances}}
\end{{deluxetable*}}
"""

print(latex)


\startdata
A1 & Cl* NGC 346 ELS 22 & $21.176 \pm 0.017$ & $15.567 \pm 0.013$ & $15.114 \pm 0.138$ & $6.390 \pm 0.022$ & $5.937 \pm 0.139$ \\
A2 & Cl* NGC 346 ELS 50 & $20.816 \pm 0.040$ & $15.403 \pm 0.011$ & $14.694 \pm 0.010$ & $6.585 \pm 0.042$ & $5.876 \pm 0.042$ \\
A3 & Cl* NGC 346 ELS 51 & $20.905 \pm 0.033$ & $15.482 \pm 0.010$ & $14.866 \pm 0.261$ & $6.576 \pm 0.035$ & $5.960 \pm 0.263$ \\
A4 & Cl* NGC 346 ELS 7 & $21.096 \pm 0.022$ & $15.602 \pm 0.018$ & $15.218 \pm 0.081$ & $6.505 \pm 0.029$ & $6.121 \pm 0.084$ \\
A5 & Cl* NGC 346 MPG 782 & $20.043 \pm 0.157$ & $15.399 \pm 0.007$ & $14.736 \pm 0.052$ & $7.348 \pm 0.158$ & $6.685 \pm 0.166$ \\
A6 & Cl* NGC 346 NMC 17 & $21.288 \pm 0.016$ & $15.791 \pm 0.035$ & $15.170 \pm 0.116$ & $6.502 \pm 0.039$ & $5.881 \pm 0.117$ \\
A7 & Cl* NGC 346 NMC 28 & $21.030 \pm 0.026$ & $15.567 \pm 0.021$ & $14.749 \pm 0.009$ & $6.536 \pm 0.034$ & $5.718 \pm 0.028$ \\
A8 & Cl* NGC 346 SSN 25 & $21.487 \pm 0.019$ & $15.686 \pm 0.027$ & $14.976 \p

In [12]:
# END